# **Background Sutract Images: CAAX channel**

#### In this notebook, we will subtract background from ome-tiff images

In [ ]:
# Import packages
import sys
import numpy as np
from pathlib import Path
src_path = str(Path.cwd().parent)
if src_path not in sys.path:
    sys.path.append(src_path)

import microscopy_analysis.d00_utils.utilities as utils
from microscopy_analysis.d01_init_proc.subtractbg import subtract_bg
import microscopy_analysis.d00_utils.dirnames as dn

%matplotlib notebook
%matplotlib inline

import pandas as pd

## Import images

##### **Add path for a folder of images to be analyzed.**
##### Example: exp_dir = "/Users/kwu2/Documents/Experiments/mRcaax594_MBP647"


In [ ]:
input_dirpath = Path(input('Please enter the path for the folder containing images:'))

In [ ]:
imgnames = [imgpath.name for imgpath in input_dirpath.glob('*.ome.tif')]
imgnames.sort()

num_imgs = len(imgnames)
print(f'{num_imgs} images found in directory')

In [ ]:
# process caax channel
ch_to_process = 0
ch_content = 'caax'

# Input initial parameters to try

# enter outlier percentiles in a list (i.e. [50, 80, 100])
outlier_percentiles = [70, 80, 85, 90, 95, 97, 100]

# enter sigmas for smoothing in a list format (i.e. [50, 80, 100])
sigmas_smoothing = [0.25]

In [ ]:
# Get output dirpaths
proc_dirpath = utils.get_proc_dirpath(input_dirpath)
bg_sub_dirpath = proc_dirpath / dn.bg_sub_dirname / (input_dirpath.name + '_bgsbch' + str(ch_to_process) + '_init')

bin_dirpath = proc_dirpath / dn.bg_sub_dirname / (input_dirpath.name + '_binch' + str(ch_to_process) + '_init')
bin_fig_dirpath = bin_dirpath / dn.figs_dirname

In [ ]:
# Create table to be used for background subtraction

bgsub_df = pd.DataFrame()
bgsub_df['input image name'] = imgnames
bgsub_df['img idx'] = np.arange(len(bgsub_df))
bgsub_df['selected param'] = 'NA'
bgsub_df['BG subtract?'] = True
bgsub_df['ch_to_process'] = ch_to_process
bgsub_df['ch content'] = ch_content
bgsub_df['outlier_percentiles'] = [outlier_percentiles] * len(bgsub_df)
bgsub_df['rescale_perc_grayval'] = [30] * len(bgsub_df)
bgsub_df['sigmas_smoothing'] = [sigmas_smoothing] * len(bgsub_df)
bgsub_df['# prev tested params'] = 0
bgsub_df['input dirname'] = input_dirpath.name
bgsub_df['init bgsb dirpath'] = bg_sub_dirpath
bgsub_df['init bin dirpath'] = bin_dirpath

bgsub_df.head()

In [ ]:
# Save background subtract dataframe
bgsub_df_name = f'bgsub_{ch_content}.csv'

proc_dirpath = utils.get_proc_dirpath(input_dirpath)
bgsub_df_path = utils.get_proc_dirpath(input_dirpath) / dn.tables_dirname / bgsub_df_name

# if (proc_dirpath / dn.tables_dirname).is_dir():
#     bgsub_df.to_csv(bgsub_df_path, index=False)

### Remove background from ome-tiff images.

##### Optimize parameters for background subtraction for each channel

In [ ]:
subtract_bg(input_dirpath, bgsub_df_path)
print('done!')